# 功率读数按小时聚合
适用于规则采样长表。净表计的正负号原样保留。无表头时可用整数列索引。

In [ ]:
from load_common import export, hourly_frame, fill_meter_gaps, PERIODS
import numpy as np
import pandas as pd

YEAR = 2026
TARIFF = "inputs/tariff.xlsx"
INPUT = "inputs/power.xlsx"
OUTPUT = "outputs/power_8760.xlsx"
SHEET = 0
HEADER = 0  # 无表头改 None
SKIPROWS = 0
TIME_COL = "数据时间"
POWER_COL = "总有功功率"
RATIO_COL = None  # 有逐记录综合倍率时填列名或整数列索引
MULTIPLIER = 1.0  # 无倍率列时使用，已还原填 1
POINT_COL = None  # 多测量点时填列名或整数列索引
TARGET_POINT = 1


In [ ]:
raw = pd.read_excel(INPUT, sheet_name=SHEET, header=HEADER, skiprows=SKIPROWS)
if POINT_COL is not None:
    raw = raw.loc[raw[POINT_COL] == TARGET_POINT]
times = pd.to_datetime(raw[TIME_COL], errors="raise")
power = pd.to_numeric(raw[POWER_COL], errors="raise")
ratio = MULTIPLIER if RATIO_COL is None else pd.to_numeric(raw[RATIO_COL], errors="raise")
samples = pd.Series((power * ratio).to_numpy(), index=pd.DatetimeIndex(times)).dropna()
if samples.index.hasnans or samples.index.has_duplicates:
    raise ValueError("检查空时间戳、重复记录和测量点筛选")
hourly = samples.resample("h").mean()

frame = hourly_frame(YEAR, TARIFF)
frame["负载"], frame["负载来源"] = fill_meter_gaps(hourly, frame.index)
audit = frame.groupby(["月份", "负载来源"], as_index=False).agg(
    小时数=("负载", "size"), 电量_kWh=("负载", "sum")
)
export(frame, OUTPUT, audit)
